# HW Tracker

Signs in to Brightspace, WebAssign and WileyPLUS, collects every assignment and its due date into one list, remembers what it saw last time, and then sends the results to a to-do note, Discord and Google Calendar.

Everything, including first-time setup, is in this notebook. Run the cells in order (or **Run All**).

**First time on a new PC:** install (see README), run sections 1 and 2, run `setup_logins()` from the Setup section at the end, then **Run All**.

```
Brightspace ─┐                                  ┌─> to-do note (Markdown)
WebAssign ───┼─> sign in, read due dates ─> list ┼─> Discord
WileyPLUS ───┘        (sections 3–7)    (sec. 8–9)└─> Google Calendar
```

| Section | What it does |
|---|---|
| 1. Settings | Loads `config.json` (writes a default one if missing) and sets the safety switches |
| 2. Passwords | Decrypts the saved logins (starts empty on a new PC) |
| 3. Browser | Opens Chrome, controlled by code (Playwright) |
| 4. Signing in | One helper that handles all three login pages |
| 5–7. Site readers | Brightspace, WileyPLUS, WebAssign |
| 8. Run the readers | Collects everything into one table |
| 9. Memory | Compares with last run: what is new, what moved |
| 10–12. Outputs | Note, Discord, Google Calendar |
| 13. Save | Writes the memory file for next time |

**Every reader returns assignments in the same shape**: `source`, `key`, `course`, `title`, `due`, `all_day`, `url`, `done`. Everything after section 8 only ever sees that list, so it does not care which site an assignment came from.

## 1. Settings

`config.json` holds the site addresses and a few preferences. The defaults are written out below in `DEFAULT_CONFIG`; **if `config.json` is missing, this cell creates it from them**. After that, the file is what counts, so you can edit it (say, a new course name) without touching the code. Delete it to go back to the defaults.

Some values are **regular expressions** (patterns for matching text), which is why they are written as `r"..."`: `readyUrl` is "what the address looks like once we are signed in", and a `hop` is "on a page whose address matches `url`, click the element `click`".

The switches below keep a run safe: by default it reads everything but writes only inside this folder.


In [ ]:
import asyncio
import base64
import hashlib
import json
import re
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlencode, urljoin, urlparse

import pandas as pd
import requests

FOLDER = Path.cwd()  # the folder this notebook is in

DEFAULT_CONFIG = {
    "lookaheadDays": 120,  # how far ahead to look for deadlines
    "sites": {
        # enabled: read this site at all. verified: a person checked its results (see section 8).
        "brightspace": {
            "enabled": True, "verified": True,
            "login": "brightspace",
            "host": "https://brightspace.uri.edu",
        },
        "webassign": {
            "enabled": True, "verified": True,
            "login": "cengage",
            "course": "PHY 203",  # WebAssign does not name the course on its page
            "startUrl": "https://www.webassign.net/wa-auth/login",
            "readyUrl": r"webassign\.net/(web/Student|v4cgi/student\.pl)",
            "hops": [{"url": r"cengage\.com/dashboard",
                      "click": "a[data-testid='home-page-launch-course-link'][href*='webassign']"}],
            "scope": "#js-student-myAssignmentsWrapper",  # the "My Assignments" box
        },
        "wileyplus": {
            "enabled": True, "verified": True,
            "login": "wiley",
            "startUrl": "https://education.wiley.com/",
            "readyUrl": r"learn\.wileyplus\.com/courses/\d+",
            "hops": [{"url": r"education\.wiley\.com/.*StudentDashboard",
                      "click": "a[data-testid='course-card__course-section-link']"}],
        },
    },
    "calendar": {
        "calendarId": "primary",
        "colorId": "5",                 # Google's yellow
        "blockMinutes": 30,             # each event is a 30-minute block ending at the deadline
        "reminderMinutes": [1440, 180], # 1 day and 3 hours before
    },
    "discord": {
        "digestHour": 8,  # the daily "due soon" message is sent on the first run after 8 AM
        "digestDays": 3,
    },
}

CONFIG_FILE = FOLDER / "config.json"
if not CONFIG_FILE.exists():
    CONFIG_FILE.write_text(json.dumps(DEFAULT_CONFIG, indent=2))
    print("No config.json yet, so the defaults were saved to", CONFIG_FILE)
config = json.loads(CONFIG_FILE.read_text())

# Safety switches. Turn them on only when this notebook should do the real job.
SEND_TO_DISCORD = False      # False = print the messages instead of posting them
WRITE_TO_CALENDAR = False    # False = say what would be added, add nothing
NOTE_PATH = FOLDER / "output" / "Homework Due.md"   # for the real thing: Path(r"C:\Users\dniki\OneDrive\Obsidian\Classes\Homework Due.md")

LOOKAHEAD_DAYS = config["lookaheadDays"]   # how far ahead to look for deadlines
DAY = timedelta(days=1)
now = datetime.now().astimezone()          # the current time, with the timezone attached
print("Run started at", now.strftime("%a %b %d, %I:%M %p"))

## 2. Passwords

The logins are stored in `secrets.json`, encrypted with **Windows DPAPI**: Windows' built-in encryption, with a key tied to this Windows account on this PC. Copied to another computer, the file is unreadable. The program never handles an encryption key itself.

`win32crypt` (from the `pywin32` package) asks Windows to decrypt it.

On a new PC there is no `secrets.json` yet, so this starts with no logins; save them with `setup_logins()` in the Setup section at the end. Because of DPAPI, each person makes their own `secrets.json` on their own PC; copying someone else's does not work.

In [ ]:
import win32crypt

def load_secrets():
    saved = json.loads((FOLDER / "secrets.json").read_text())
    encrypted = base64.b64decode(saved["blob"])
    _, decrypted = win32crypt.CryptUnprotectData(encrypted, None, None, None, 0)
    return json.loads(decrypted)

if (FOLDER / "secrets.json").exists():
    secrets = load_secrets()
else:
    secrets = {"logins": {}}
    print("No secrets.json yet: run setup_logins() in the Setup section at the end.")
print("Saved logins:", ", ".join(secrets["logins"]) or "none")   # site names only, never the passwords
print("Discord connected:", "discordWebhook" in secrets)
print("Google Calendar connected:", "google" in secrets)

## 3. Browser

**Playwright** controls a real Chrome browser from code: open pages, type, click, read.

- The **profile** folder keeps cookies between runs, the same way your own browser stays logged in. That is why Microsoft rarely asks for MFA again.
- `headless=False` shows the window, so you can watch it work (and approve MFA on your phone if asked).
- `await` means "wait until the browser has finished this step". Browser actions take time, so most lines that touch the browser start with `await`.
- `route(...)` blocks one pop-up service, **Pendo**. In Nov 2026 WileyPLUS started showing a survey lightbox through it that covered the Login button, and the tracker failed for a week until it was blocked.

In [ ]:
from playwright.async_api import async_playwright

playwright = await async_playwright().start()
browser = await playwright.chromium.launch_persistent_context(
    str(FOLDER / "profile"),
    headless=False,
    viewport={"width": 1280, "height": 900},
)
# Pendo shows pop-ups (WileyPLUS's "take a survey" lightbox, Nov 2026) that cover
# the page and swallow every click. Refuse to load it at all.
await browser.route(re.compile(r"^https?://[^/]*pendo\.io/"), lambda route: route.abort())
print("Browser open")

## 4. Signing in

All three sites log in differently (Microsoft for Brightspace, Cengage for WebAssign, Wiley for WileyPLUS), and the pages do not always appear in the same order. So instead of a fixed script, `sign_in` is a **loop**. Every 1.5 seconds it looks at every open tab and asks:

1. Are we signed in yet? → done.
2. Is there a link we were told to click on the way (a "hop", e.g. *Open WebAssign* on the Cengage dashboard)? → click it.
3. Is there a login form? → fill it in.

Two safety details:
- **Only boxes a person could click count.** Microsoft hides a tiny 10-pixel password box on its email page; typing into it broke the first version.
- **One try per site.** A wrong password is never typed twice, so the program cannot lock the account.

In [ ]:
NOT_A_TEXT_BOX = ":not([type=checkbox]):not([type=radio]):not([type=hidden]):not([type=submit]):not([type=button])"
EMAIL_BOXES = ", ".join(
    selector + NOT_A_TEXT_BOX
    for selector in [
        "input[type=email]", "input[name=loginfmt]", "input[autocomplete=username]",
        "input[name*=user i]", "input[id*=user i]", "input[name*=email i]", "input[id*=email i]",
    ]
)
MICROSOFT = ("login.microsoftonline.com", "login.microsoft.com", "login.live.com")


async def clickable(frame, selector):
    """The boxes matching `selector` that are big enough and on screen."""
    boxes = []
    for element in await frame.locator(selector).all():
        size = await element.bounding_box()
        if size and size["width"] >= 40 and size["height"] >= 15 and size["x"] >= 0 and size["y"] >= 0:
            boxes.append(element)
    return boxes


async def press_submit(frame, box):
    buttons = await clickable(frame, "button[type=submit], input[type=submit]")
    if len(buttons) == 1:
        await buttons[0].click()
    else:
        await box.press("Enter")


async def try_login(frame, login, tried):
    """Fills in a login form if this frame shows one. Returns True if it typed something."""
    host = urlparse(frame.url).netloc
    passwords = await clickable(frame, "input[type=password]")
    emails = await clickable(frame, EMAIL_BOXES)

    # One password box = a sign-in form (two would be a sign-up form).
    if len(passwords) == 1 and (host, "password") not in tried:
        if emails and not await emails[0].input_value():
            await emails[0].fill(login["user"])
        await passwords[0].fill(login["password"])
        await press_submit(frame, passwords[0])
        tried.add((host, "password"))
        return True

    # Email-only page: Microsoft asks for the email first and the password on the next page.
    if not passwords and emails and (host, "email") not in tried:
        await emails[0].fill(login["user"])
        await press_submit(frame, emails[0])
        tried.add((host, "email"))
        return True

    # Microsoft's "Stay signed in?" page: click Yes so the login lasts until next run.
    if host in MICROSOFT and (host, "stay") not in tried:
        if await frame.get_by_text("Stay signed in?").count():
            await frame.locator("#idSIButton9, [data-testid=primaryButton]").first.click()
            tried.add((host, "stay"))
            return True
    return False


async def sign_in(name, is_ready, login, hops=(), timeout_seconds=180):
    """Waits until some tab passes `is_ready`, filling in login forms on the way."""
    tried = set()
    give_up_at = time.time() + timeout_seconds
    while time.time() < give_up_at:
        for page in browser.pages:
            try:
                if await is_ready(page):
                    return page
                for hop in hops:
                    if hop["click"] not in tried and re.search(hop["url"], page.url, re.I):
                        link = page.locator(hop["click"]).first
                        if await link.is_visible():
                            tried.add(hop["click"])
                            await link.click()
                for frame in page.frames:
                    if await try_login(frame, login, tried):
                        break
            except Exception:
                pass  # the page changed while we were reading it; look again next round
        await asyncio.sleep(1.5)
    stuck_at = ", ".join(page.url.split("?")[0] for page in browser.pages)
    raise TimeoutError(f"{name}: not signed in after {timeout_seconds}s (stuck at {stuck_at})")

## Two helpers every reader uses

- `api_get`: many sites have an internal **API**: web addresses that answer with raw data (**JSON**) instead of a page. Because the browser is signed in, its cookies make the site treat these requests as you. No API key needed.
- `course_code`: pulls the short course name out of a long one.

In [ ]:
async def api_get(url):
    """Asks a site's API for data, using the browser's login cookies."""
    response = await browser.request.get(url)
    if not response.ok:
        raise RuntimeError(f"API error {response.status} for {url.split('?')[0]}")
    text = await response.text()
    return json.loads(text.removeprefix("while(1);"))  # Canvas puts this guard in front of its JSON


def course_code(text):
    """'EGR101 0005 Fall 2026 Intro to Engineering' -> 'EGR 101'"""
    match = re.search(r"\b([A-Z]{2,4})[\s-]?(\d{3}[A-Z]?)\b", text or "")
    return f"{match.group(1)} {match.group(2)}" if match else None


def utc_text(moment):
    """A time written the way these APIs expect it: 2026-11-09T21:00:00.000Z"""
    return moment.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")


print(course_code("EGR101 0005 Fall 2026 Intro to Engineering"))

## 5. Brightspace (read through its API)

After signing in, three questions to Brightspace's API:

1. **Which API version do you run?** Asked instead of hard-coded, so a Brightspace update does not break this.
2. **Which courses am I in?** The answer comes in pages; `bookmark` says where the next page starts. Courses that have ended are skipped.
3. **Every calendar event in those courses**, from a week ago to `LOOKAHEAD_DAYS` ahead.

Then it cleans up: "available from" events (types 2 and 4) are not deadlines, so they are dropped, and " - Due" is cut off the titles.

In [ ]:
async def read_brightspace():
    host = config["sites"]["brightspace"]["host"]
    page = await browser.new_page()
    await page.goto(host + "/d2l/home")

    async def signed_in(p):
        if not p.url.startswith(host + "/d2l/"):
            return False
        answer = await browser.request.get(host + "/d2l/api/lp/1.0/users/whoami")
        return answer.ok

    await sign_in("brightspace", signed_in, secrets["logins"][config["sites"]["brightspace"]["login"]])

    # 1. API versions
    versions = await api_get(host + "/d2l/api/versions/")
    latest = {v["ProductCode"]: v["LatestVersion"] for v in versions}

    # 2. My courses
    courses = {}
    bookmark = ""
    while True:
        batch = await api_get(
            f"{host}/d2l/api/lp/{latest['lp']}/enrollments/myenrollments/?orgUnitTypeId=3&bookmark={bookmark}"
        )
        for row in batch["Items"]:
            access = row.get("Access") or {}
            ended = access.get("EndDate") and datetime.fromisoformat(access["EndDate"]) < now
            if access.get("CanAccess") and not ended:
                courses[row["OrgUnit"]["Id"]] = row["OrgUnit"]["Name"]
        paging = batch.get("PagingInfo") or {}
        if not paging.get("HasMoreItems"):
            break
        bookmark = paging["Bookmark"]

    # 3. Calendar events in those courses
    query = urlencode({
        "orgUnitIdsCSV": ",".join(str(course_id) for course_id in courses),
        "startDateTime": utc_text(now - 7 * DAY),
        "endDateTime": utc_text(now + LOOKAHEAD_DAYS * DAY),
    })
    result = await api_get(f"{host}/d2l/api/le/{latest['le']}/calendar/events/myEvents/?{query}")
    events = result if isinstance(result, list) else (result.get("Objects") or result.get("Items") or [])
    await page.close()

    items = []
    for event in events:
        if event["EventType"] in (2, 4):  # "available from" / "unlocks": not deadlines
            continue
        title = event.get("Title") or ""
        if re.search(r" - (Available|Availability Starts)$", title, re.I):
            continue
        title = re.sub(r" - (Due|Availability Ends)$", "", title, flags=re.I).strip()

        due = datetime.fromisoformat(event.get("EndDateTime") or event["StartDateTime"]).astimezone()
        all_day = bool(event.get("IsAllDayEvent"))
        if all_day:
            due = due.replace(hour=23, minute=59, second=0, microsecond=0)

        course_name = courses.get(event["OrgUnitId"])
        link = (event.get("AssociatedEntity") or {}).get("Link") or event.get("CalendarEventViewUrl")
        items.append({
            "source": "brightspace",
            "key": str(event["CalendarEventId"]),
            "course": course_code(course_name) or course_name or "Brightspace",
            "title": title,
            "due": due,
            "all_day": all_day,
            "url": urljoin(host, link) if link else None,
            "done": False,
        })
    return items

## 6. WileyPLUS (read through its API)

WileyPLUS turned out to be built on **Canvas**, a common course platform with a well-documented API. So after signing in it asks Canvas for the courses, then every assignment in each course **with its submission record**. That is how WileyPLUS items tick themselves off as done. It is the only site that reports this.

In [ ]:
async def read_wileyplus():
    site = config["sites"]["wileyplus"]
    start = await browser.new_page()
    await start.goto(site["startUrl"])

    async def on_course_page(p):
        return re.search(site["readyUrl"], p.url, re.I) is not None

    page = await sign_in("wileyplus", on_course_page, secrets["logins"][site["login"]], site["hops"])
    address = urlparse(page.url)
    origin = f"{address.scheme}://{address.netloc}"

    items = []
    courses = await api_get(origin + "/api/v1/courses?enrollment_state=active&per_page=100")
    for course in courses:
        assignments = await api_get(
            f"{origin}/api/v1/courses/{course['id']}/assignments?per_page=100&include[]=submission"
        )
        for assignment in assignments:
            if not assignment["due_at"]:
                continue
            submission = assignment.get("submission") or {}
            items.append({
                "source": "wileyplus",
                "key": str(assignment["id"]),
                "course": site.get("course") or course_code(course["name"]) or course["name"],
                "title": assignment["name"],
                "due": datetime.fromisoformat(assignment["due_at"]).astimezone(),
                "all_day": False,
                "url": assignment["html_url"],
                "done": bool(submission.get("submitted_at")),
            })

    await page.close()
    if not start.is_closed():
        await start.close()
    return items

## 7. WebAssign (read from the page)

WebAssign has no API we can use, so this reader works on the page itself. Luckily, the *My Assignments* box carries its whole list as data, in an attribute called `data-assignments`: each assignment's name and its due time as a **Unix timestamp** (seconds since 1 Jan 1970). So there is no date text to guess at.

In [ ]:
async def read_webassign():
    site = config["sites"]["webassign"]
    start = await browser.new_page()
    await start.goto(site["startUrl"])

    async def on_student_page(p):
        return any(re.search(site["readyUrl"], frame.url, re.I) for frame in p.frames)

    page = await sign_in("webassign", on_student_page, secrets["logins"][site["login"]], site["hops"])

    box = page.locator(site["scope"])          # the "My Assignments" box
    await box.wait_for(state="attached", timeout=30_000)
    data = json.loads(await box.get_attribute("data-assignments"))

    items = []
    for assignment in data["current_assignments"]:
        seconds = int(assignment["due"]["seconds"])
        items.append({
            "source": "webassign",
            "key": None,  # WebAssign gives no stable id here, so course + title is used instead
            "course": site["course"],
            "title": assignment["name"],
            "due": datetime.fromtimestamp(seconds, tz=timezone.utc).astimezone(),
            "all_day": False,
            "url": None,
            "done": False,
        })

    await page.close()
    if not start.is_closed():
        await start.close()
    return items

## 8. Run the readers

Each site runs inside its own `try`, so if one breaks, the others still work and the error is kept for the Discord alert.

A site is skipped if no login has been saved for it yet. It also only runs if `config.json` marks it `enabled` and `verified`. *Verified* is the "check, then trust" gate: a new reader prints what it finds until a person has looked and approved it, so a buggy reader cannot fill the calendar with junk.

In [ ]:
readers = {
    "brightspace": read_brightspace,
    "webassign": read_webassign,
    "wileyplus": read_wileyplus,
}

found = []
errors = []
for name, reader in readers.items():
    site = config["sites"][name]
    if not (site["enabled"] and site["verified"]):
        continue
    if site["login"] not in secrets["logins"]:
        print(f"{name}: skipped, no login saved (run setup_logins())")
        continue
    try:
        items = await reader()
        print(f"{name}: {len(items)} item(s)")
        found += items
    except Exception as error:
        errors.append(f"{name}: {error}")
        print(f"{name}: FAILED {error}")

await browser.close()
await playwright.stop()

table = pd.DataFrame(found, columns=["course", "title", "due", "done", "source"]).sort_values("due")
table["due"] = table["due"].map(lambda d: d.strftime("%a %b %d, %I:%M %p"))
table

## 9. Memory: what is new, what moved

`state.json` is what the program remembers between runs.

Each assignment gets an **id**: a short fingerprint (a SHA-1 hash) of its site and its id on that site. **The due date is left out on purpose.** When a professor moves a deadline, the id stays the same, so the program sees *the same assignment with a new date*: it reports "due date changed" and moves the calendar event instead of adding a second one.

A site can mark an item done, but never un-done: a tick you made yourself is never removed.

In [ ]:
STATE_FILE = FOLDER / "state.json"
state = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {"items": {}}


def make_id(item):
    key = item["key"] or f"{item['course']}|{item['title'].lower()}"
    return hashlib.sha1(f"{item['source']}|{key}".encode()).hexdigest()[:10]


added = []     # assignments never seen before
changed = []   # (assignment, old due date) pairs
for item in found:
    item_id = make_id(item)
    due = item["due"].astimezone(timezone.utc).isoformat()
    fresh = {
        "source": item["source"], "course": item["course"], "title": item["title"],
        "due": due, "all_day": item["all_day"],
    }
    known = state["items"].get(item_id)
    if known is None:
        state["items"][item_id] = {"id": item_id, **fresh, "url": item["url"], "done": False,
                                   "first_seen": now.isoformat()}
        if item["due"] > now:
            added.append(state["items"][item_id])
    else:
        if known["due"] != due:
            changed.append((known, known["due"]))
        known.update(fresh)
        known["url"] = item["url"] or known.get("url")
    if item["done"]:
        state["items"][item_id]["done"] = True
    state["items"][item_id]["last_seen"] = now.isoformat()

print(f"{len(added)} new, {len(changed)} changed, {len(state['items'])} remembered in total")

## 10. Output: the to-do note

The note is rebuilt from scratch every run, but your ticks survive: every line ends with the assignment's id hidden in an Obsidian comment (`%%id%%`), and ticked lines are read back *before* the note is rewritten.

In [ ]:
def when(item):
    """'Wed Nov 11, 10:00 PM' (or just 'Wed Nov 11' for an all-day item)"""
    due = datetime.fromisoformat(item["due"]).astimezone()
    day = f"{due:%a %b} {due.day}"
    return day if item["all_day"] else f"{day}, {due:%I:%M %p}".replace(", 0", ", ")


def write_note(path):
    path.parent.mkdir(parents=True, exist_ok=True)

    # Read back the boxes ticked by hand, so the rewrite keeps them.
    if path.exists():
        for item_id in re.findall(r"^- \[x\] .*%%(\w+)%%\s*$", path.read_text(encoding="utf-8"), re.I | re.M):
            if item_id in state["items"]:
                state["items"][item_id]["done"] = True

    items = sorted(state["items"].values(), key=lambda item: item["due"])
    days_left = lambda item: (datetime.fromisoformat(item["due"]) - now) / DAY

    def line(item):
        box = "x" if item["done"] else " "
        title = f"[{item['title']}]({item['url']})" if item.get("url") else item["title"]
        return f"- [{box}] **{when(item)}** · {item['course']} · {title} %%{item['id']}%%"

    def section(heading, chosen):
        return [f"## {heading}", "", *map(line, chosen), ""] if chosen else []

    open_items = [item for item in items if not item["done"]]
    lines = [
        "---", "tags: [homework, auto-generated]", f"updated: {now:%Y-%m-%d %H:%M}", "---", "",
        "# Homework Due", "",
        "> [!info] Written by hw-tracker",
        "> This note is rebuilt on every sync. Tick a box when something is done and it stays ticked; "
        "any other edit is overwritten.", "",
        *section("Past due — tick if done", [i for i in open_items if -7 <= days_left(i) < 0]),
        *section("Next 7 days", [i for i in open_items if 0 <= days_left(i) < 7]),
        *section("Later", [i for i in open_items if days_left(i) >= 7]),
        *section("Done", [i for i in items if i["done"] and days_left(i) >= -14]),
    ]
    path.write_text("\n".join(lines), encoding="utf-8")


write_note(NOTE_PATH)
print("Note written to", NOTE_PATH)

## 11. Output: Discord

A Discord **webhook** is a secret web address; anything sent to it appears as a message in one channel. No bot needed.

What gets sent:
- new assignments, and due dates that changed;
- a **daily digest** of what is due in the next 3 days, sent even when nothing is due. *No message one morning means the tracker itself is down* (a "dead-man's switch");
- a warning when a site fails, once when it starts, then daily until it is fixed, then "working again".

`<t:…:R>` is Discord's own timestamp format: it shows "in 2 days" in each reader's timezone.

In [ ]:
def discord_line(item):
    seconds = int(datetime.fromisoformat(item["due"]).timestamp())
    return f"• **{item['course']}** {item['title']} — <t:{seconds}:f> (<t:{seconds}:R>)"


def post_to_discord(heading, lines=()):
    message = heading
    for count, text in enumerate(lines):
        if len(message) + len(text) + 1 > 1900:  # Discord rejects messages over 2000 characters
            message += f"\n…and {len(lines) - count} more"
            break
        message += "\n" + text
    if not SEND_TO_DISCORD or "discordWebhook" not in secrets:
        why = "switched off" if not SEND_TO_DISCORD else "not connected (run setup_discord())"
        print(f"(Discord is {why}; this would be sent)\n" + message + "\n")
        return
    answer = requests.post(secrets["discordWebhook"], json={"content": message, "allowed_mentions": {"parse": []}})
    answer.raise_for_status()


if added:
    post_to_discord(f"📚 **{len(added)} new assignment(s)**", [discord_line(item) for item in added])
if changed:
    post_to_discord("🔁 **Due date changed**", [f"{discord_line(item)} (was {when({**item, 'due': old})})" for item, old in changed])

digest = config["discord"]
today = now.strftime("%Y-%m-%d")
if state.get("lastDigest") != today and now.hour >= digest["digestHour"]:
    soon = [
        item for item in sorted(state["items"].values(), key=lambda item: item["due"])
        if not item["done"] and now < datetime.fromisoformat(item["due"]) < now + digest["digestDays"] * DAY
    ]
    if soon:
        post_to_discord(f"⏰ **Due in the next {digest['digestDays']} days**", [discord_line(item) for item in soon])
    else:
        post_to_discord(f"✅ Nothing due in the next {digest['digestDays']} days")
    state["lastDigest"] = today

failure = "\n".join(errors)
if failure and failure != state.get("lastFailure"):
    post_to_discord("⚠️ **hw-tracker needs attention**", errors)
if not failure and state.get("lastFailure"):
    post_to_discord("✅ hw-tracker is working again")
state["lastFailure"] = failure

## 12. Output: Google Calendar

**Connecting** uses **OAuth**: you approve access once in your browser, and Google hands back a *refresh token*, which is saved (encrypted) with the passwords. It only allows managing calendar events, nothing else in the Google account, and the program never sees your Google password. The one-time approval is `setup_google()` in the **Setup** section at the end.

**Every run** trades the refresh token for a short-lived *access token* (good for about an hour), then for each upcoming assignment:

1. Look for an event already covering it: one this program made before (found by a hidden `hwTrackerId` tag on the event), or one **you added by hand** for the same class ending at the same time. Then *adopt* it instead of adding a duplicate.
2. Otherwise create one: `PHY 203 HW8 due`, a 30-minute yellow block that ends at the deadline, shown as *free*, with reminders 1 day and 3 hours before.
3. If a deadline moved, move the event. It never deletes anything.

In [ ]:
CALENDAR = config["calendar"]
EVENTS_URL = f"https://www.googleapis.com/calendar/v3/calendars/{CALENDAR['calendarId']}/events"


def google_access_token():
    google = secrets["google"]
    answer = requests.post("https://oauth2.googleapis.com/token", data={
        "client_id": google["clientId"],
        "client_secret": google["clientSecret"],
        "refresh_token": google["refreshToken"],
        "grant_type": "refresh_token",
    })
    answer.raise_for_status()
    return answer.json()["access_token"]


def event_title(item):
    """'PHY 203' + 'HW8' -> 'PHY 203 HW8 due'"""
    squash = lambda text: text.replace(" ", "").lower()
    name = item["title"] if squash(item["title"]).startswith(squash(item["course"])) else f"{item['course']} {item['title']}"
    return name if name.lower().endswith(" due") else f"{name} due"


def event_body(item):
    due = datetime.fromisoformat(item["due"])
    reminders = CALENDAR["reminderMinutes"][:1] if item["all_day"] else CALENDAR["reminderMinutes"]
    body = {
        "summary": event_title(item),
        "description": "\n".join(filter(None, [item.get("url"), f"Source: {item['source']}", "Added by hw-tracker."])),
        "colorId": CALENDAR["colorId"],
        "transparency": "transparent",  # shown as "free", so it does not block the schedule
        "extendedProperties": {"private": {"hwTrackerId": item["id"]}},
        "reminders": {"useDefault": False, "overrides": [{"method": "popup", "minutes": m} for m in reminders]},
    }
    if item["all_day"]:
        local_day = due.astimezone().date()
        body["start"] = {"date": local_day.isoformat()}
        body["end"] = {"date": (local_day + DAY).isoformat()}
    else:
        body["start"] = {"dateTime": (due - timedelta(minutes=CALENDAR["blockMinutes"])).isoformat()}
        body["end"] = {"dateTime": due.isoformat()}
    return body


def find_existing(item, headers):
    """An event that already covers this assignment, and whether it was made by hand."""
    own = requests.get(EVENTS_URL, headers=headers, params={"privateExtendedProperty": f"hwTrackerId={item['id']}"}).json()
    if own.get("items"):
        return own["items"][0], False
    if item["all_day"]:
        return None, False

    due = datetime.fromisoformat(item["due"])
    nearby = requests.get(EVENTS_URL, headers=headers, params={
        "timeMin": (due - timedelta(hours=2)).isoformat(),
        "timeMax": (due + timedelta(hours=2)).isoformat(),
        "singleEvents": "true",
    }).json().get("items", [])
    squash = lambda text: text.replace(" ", "").lower()
    same_class = [
        event for event in nearby
        if event.get("status") != "cancelled"
        and "dateTime" in event.get("end", {})
        and abs(datetime.fromisoformat(event["end"]["dateTime"]) - due) <= timedelta(minutes=5)
        and squash(event.get("summary", "")).startswith(squash(item["course"]))
    ]
    exact = [event for event in same_class if squash(item["title"]) in squash(event["summary"])]
    if exact:
        return exact[0], True
    if len(same_class) == 1:
        return same_class[0], True
    return None, False


def sync_calendar():
    headers = {"Authorization": f"Bearer {google_access_token()}"}
    created = moved = 0
    for item in state["items"].values():
        if item.get("calendar_gone"):
            continue
        body = event_body(item)
        if not item.get("event_id"):
            if datetime.fromisoformat(item["due"]) < now:
                continue
            event, by_hand = find_existing(item, headers)
            if event:
                item["event_id"], item["adopted"] = event["id"], by_hand
            else:
                item["event_id"] = requests.post(EVENTS_URL, headers=headers, json=body).json()["id"]
                created += 1
            item["synced_due"] = item["due"]
        elif item.get("synced_due") != item["due"]:
            change = {"start": body["start"], "end": body["end"]} if item.get("adopted") else body
            answer = requests.patch(f"{EVENTS_URL}/{item['event_id']}", headers=headers, json=change)
            if answer.status_code in (404, 410) or answer.json().get("status") == "cancelled":
                item["calendar_gone"] = True  # deleted by hand: respect that
            else:
                moved += 1
            item["synced_due"] = item["due"]
    return created, moved


if "google" not in secrets:
    print("Google Calendar is not connected yet (run setup_google() in the Setup section at the end).")
elif not WRITE_TO_CALENDAR:
    upcoming = [item for item in state["items"].values() if datetime.fromisoformat(item["due"]) > now]
    print(f"Calendar is switched off; {len(upcoming)} upcoming item(s) would be checked.")
else:
    try:
        created, moved = sync_calendar()
        print(f"calendar: {created} created, {moved} moved")
    except Exception as error:
        errors.append(f"calendar: {error}")
        print(f"calendar: FAILED {error}")

## 13. Save

Write the memory file so the next run knows what it has already seen.

In [ ]:
STATE_FILE.write_text(json.dumps(state, indent=2))
print(f"done: {len(added)} new, {len(changed)} changed, {len(errors)} error(s)")

## Setup (only when something changes)

**Run All skips this section on purpose**: the calls in the last cell are commented out. Use it to:

- save or change a site login: `setup_logins()`
- connect a Discord channel: `setup_discord()`
- connect Google Calendar: `setup_google()`

Run sections 1 and 2 first (settings and passwords; on a new PC they create `config.json` and start with no logins), then remove the `#` in front of the one you need, run that cell, and put the `#` back. Each one asks its questions in a box at the top of VS Code. Passwords are typed into a hidden box (`getpass`), so they never show up on screen or in the notebook.

Everything is saved back into `secrets.json`, encrypted with DPAPI exactly as in section 2.

### Before `setup_google()`: make an OAuth client (about 5 minutes, free)

1. Open [console.cloud.google.com](https://console.cloud.google.com), create a project, and enable the **Google Calendar API**.
2. Under **OAuth consent screen**, choose *External*, add yourself as a user, then **publish the app** ("In production"). Left in "Testing", Google expires the sign-in every 7 days.
3. Under **Credentials**, create an **OAuth client ID** of type **Desktop app**.
4. Run `setup_google()`, paste the client ID and secret, and click **Allow** in the browser tab that opens.

How the approval comes back: the notebook starts a tiny web server on this PC (`127.0.0.1`). After you click Allow, Google sends your browser to that address with a one-time **code**, and the notebook trades the code for the refresh token. The random `state` value is checked on the way back, so a fake redirect cannot slip in.

### Discord

In your server: channel settings > Integrations > Webhooks > New Webhook > Copy Webhook URL, then run `setup_discord()` and paste it. A test message is sent.

In [ ]:
import http.server
import os
import webbrowser
from getpass import getpass
from urllib.parse import parse_qs


def save_secrets(saved):
    """Encrypts the logins with DPAPI and writes secrets.json (the reverse of load_secrets)."""
    encrypted = win32crypt.CryptProtectData(json.dumps(saved).encode("utf-8"), None, None, None, None, 0)
    (FOLDER / "secrets.json").write_text(json.dumps({"blob": base64.b64encode(encrypted).decode()}, indent=2))


LOGINS = {
    "brightspace": "Brightspace (school login)",
    "cengage": "Cengage / WebAssign",
    "wiley": "WileyPLUS",
    "mcgrawhill": "McGraw Hill Connect",
}


def setup_logins():
    saved = load_secrets() if (FOLDER / "secrets.json").exists() else {}
    saved.setdefault("logins", {})
    for key, label in LOGINS.items():
        current = saved["logins"].get(key, {}).get("user")
        user = input(f"{label} email/username" + (f" [saved: {current}, blank keeps it]" if current else " (blank skips)") + ": ").strip()
        if not user:
            continue
        saved["logins"][key] = {"user": user, "password": getpass(f"{label} password (hidden): ")}
    save_secrets(saved)
    print("Logins saved:", ", ".join(saved["logins"]))


def setup_discord():
    saved = load_secrets() if (FOLDER / "secrets.json").exists() else {}
    url = getpass("Discord webhook URL (hidden, blank keeps the saved one): ").strip()
    if not url:
        print("Kept the saved webhook.")
        return
    answer = requests.post(url, json={"content": "👋 hw-tracker is connected. Alerts and the daily digest will show up here."})
    answer.raise_for_status()
    saved["discordWebhook"] = url
    save_secrets(saved)
    print("Webhook saved; a test message was sent to the channel.")


def setup_google(wait_minutes=5):
    saved = load_secrets() if (FOLDER / "secrets.json").exists() else {}
    google = saved.get("google", {})
    client_id = input("Client ID (blank keeps the saved one): ").strip() or google.get("clientId")
    if not client_id:
        print("Skipped: no client ID.")
        return
    client_secret = getpass("Client secret (hidden, blank keeps the saved one): ").strip() or google.get("clientSecret")

    # 1. A tiny web server on this PC, waiting for Google to send the browser back.
    reply = {}

    class CatchReply(http.server.BaseHTTPRequestHandler):
        def do_GET(self):
            reply.update(parse_qs(urlparse(self.path).query))
            self.send_response(200)
            self.end_headers()
            self.wfile.write(b"hw-tracker is connected. You can close this tab.")

        def log_message(self, *args):
            pass  # keep the notebook output clean

    server = http.server.HTTPServer(("127.0.0.1", 0), CatchReply)  # port 0 = any free port
    server.timeout = 5
    redirect_uri = f"http://127.0.0.1:{server.server_port}"
    state_check = os.urandom(16).hex()

    # 2. Send you to Google's "Allow access?" page.
    consent_url = "https://accounts.google.com/o/oauth2/v2/auth?" + urlencode({
        "client_id": client_id,
        "redirect_uri": redirect_uri,
        "response_type": "code",
        "scope": "https://www.googleapis.com/auth/calendar.events",  # calendar events only
        "access_type": "offline",   # ask for a refresh token
        "prompt": "consent",
        "state": state_check,
    })
    print("Approve calendar access in the browser. If nothing opened, go to:\n" + consent_url)
    webbrowser.open(consent_url)

    # 3. Wait for the reply that carries our state value.
    give_up_at = time.time() + wait_minutes * 60
    while reply.get("state") != [state_check]:
        if time.time() > give_up_at:
            server.server_close()
            raise TimeoutError("No answer from Google; run setup_google() again.")
        reply.clear()
        server.handle_request()
    server.server_close()
    if "code" not in reply:
        raise RuntimeError(f"Google said: {reply.get('error', ['no code'])[0]}")

    # 4. Trade the one-time code for a refresh token, and save it.
    answer = requests.post("https://oauth2.googleapis.com/token", data={
        "code": reply["code"][0],
        "client_id": client_id,
        "client_secret": client_secret,
        "redirect_uri": redirect_uri,
        "grant_type": "authorization_code",
    })
    answer.raise_for_status()
    saved["google"] = {"clientId": client_id, "clientSecret": client_secret, "refreshToken": answer.json()["refresh_token"]}
    save_secrets(saved)
    print("Google Calendar connected. Set WRITE_TO_CALENDAR = True in section 1 to start adding events.")

In [ ]:
# Remove the # in front of the one you need, run this cell, then put the # back.
# setup_logins()
# setup_discord()
# setup_google()